# Mempools and Oracles — When the Chain Does Not Know Yet

A blockchain can make a shared history hard to rewrite. It cannot learn a payment, a football score, or a market price until somebody sends that information into the network. This notebook starts one step earlier: with the different, incomplete views held by network participants.


## Recap

| Notebook | What it established |
| --- | --- |
| 1 | Hash links make an edited history detectable; proof of work makes rewriting costly. |
| 2 | Proof of stake schedules and rewards validators, with stake at risk for bad behaviour. |
| 3 | Merkle trees commit to many records while keeping membership proofs small. |
| 4 | Permissioned chains change who may participate, not the need for shared rules. |

This notebook adds the network layer around those ideas. **Agreement does not create information.** Nodes must first receive a transaction or a block, and that arrival is never perfectly simultaneous.


## 1. One network, many mempools

A mempool is not one global waiting room. It is each node's local list of valid transactions it has heard about so far. Two honest nodes can therefore have different mempools at the same moment.


In [ ]:
from dataclasses import dataclass
import hashlib
import json
import random
import statistics


@dataclass(frozen=True)
class Transaction:
    tx_id: str
    description: str


class Network:
    def __init__(self, node_names: list[str], rng: random.Random) -> None:
        if not node_names:
            raise ValueError("A network needs at least one node.")
        self.node_names = list(node_names)
        self.rng = rng
        self.mempools: dict[str, dict[str, Transaction]] = {
            name: {} for name in node_names
        }

    def broadcast(self, transaction: Transaction, origin: str) -> None:
        if origin not in self.mempools:
            raise ValueError(f"Unknown origin node: {origin}")
        self.mempools[origin][transaction.tx_id] = transaction
        for name in self.node_names:
            if name != origin and self.rng.random() < 0.5:
                self.mempools[name][transaction.tx_id] = transaction

    def mempool_ids(self, node_name: str) -> list[str]:
        if node_name not in self.mempools:
            raise ValueError(f"Unknown node: {node_name}")
        return sorted(self.mempools[node_name])


## 2. Gossip: useful precisely because it is imperfect

Gossip sends a transaction outward to many peers. It improves the chance that a future proposer has the transaction, but it does not promise that every peer has it immediately. We use a seeded random generator so this small demonstration is repeatable.

> Pause and predict: after four broadcasts, will every node have all four transaction IDs?


In [ ]:
nodes = ["NUS-Node", "SketchyGuy-Node", "Emma-Node", "Farid-Node"]
network = Network(nodes, random.Random(7))
transactions = [
    Transaction("tx1", "Alice pays Bob"),
    Transaction("tx2", "Carol pays Titus"),
    Transaction("tx3", "Titus pays Alice"),
    Transaction("tx4", "Bob pays Carol"),
]
origins = ["NUS-Node", "SketchyGuy-Node", "Emma-Node", "NUS-Node"]

for transaction, origin in zip(transactions, origins):
    network.broadcast(transaction, origin)

print("Each node has its own mempool:")
for node in nodes:
    print(f"  {node}: {network.mempool_ids(node)}")

mempool_sizes = [len(network.mempool_ids(node)) for node in nodes]
print(f"Average local mempool size: {statistics.mean(mempool_sizes):.2f}")


The four states are deliberately different: **origin** means the sender gave a node the transaction; **gossip receipt** means another node heard it; **inclusion** means a proposer put it in a block; and **confirmation** means the network later treats that block as part of its accepted history. A transaction can be at any earlier state without reaching the later ones.


## 3. From transactions to block proposals

A proposer can only choose from its local view. In a real chain it also checks signatures, fees, execution rules, and block limits; this notebook keeps only transaction IDs so we can see how incomplete propagation affects the next block.


## 4. Slots: one scheduled proposer, no mining race

Proof of stake divides time into slots and assigns one eligible proposer to each slot. The assignment must be reproducible by all validators, so this model hashes a documented epoch seed and slot number. It is a teaching stand-in for a real protocol's randomness and eligibility proofs.


In [ ]:
@dataclass(frozen=True)
class Validator:
    name: str
    stake: int


@dataclass
class Block:
    slot: int
    transactions: list[str]
    parent_hash: str
    proposer: str

    def __post_init__(self) -> None:
        payload = json.dumps(
            {
                "slot": self.slot,
                "transactions": self.transactions,
                "parent_hash": self.parent_hash,
                "proposer": self.proposer,
            },
            sort_keys=True,
        )
        self.hash = hashlib.sha256(payload.encode()).hexdigest()


def assign_proposer(
    validators: list[Validator], slot: int, epoch_seed: str
) -> Validator:
    if not validators or sum(v.stake for v in validators) <= 0:
        raise ValueError("Proposer assignment needs positive validator stake.")
    seed = hashlib.sha256(f"{epoch_seed}:{slot}".encode()).digest()
    pick = int.from_bytes(seed, "big") % sum(v.stake for v in validators)
    cumulative = 0
    for validator in validators:
        cumulative += validator.stake
        if pick < cumulative:
            return validator
    raise RuntimeError("Unreachable proposer-selection state.")


## 5. A fork caused by delay, not dishonesty

We use the stable epoch seed `notebook-5`. With the four validators below it assigns Farid-Node to slot 1 and NUS-Node to slot 2. They are different scheduled proposers. The slot 2 proposer has not received Block #1, so it honestly builds on the latest tip it knows: Genesis. Both candidate blocks therefore point to Genesis.

> Pause and predict: if the next proposer has not seen a valid earlier block, can deterministic proposer assignment alone prevent a fork?


In [ ]:
validators = [
    Validator("NUS-Node", 100),
    Validator("SketchyGuy-Node", 80),
    Validator("Emma-Node", 70),
    Validator("Farid-Node", 50),
]
validators_by_name = {validator.name: validator for validator in validators}
epoch_seed = "notebook-5"

genesis = Block(0, [], "0" * 64, "network")
slot_1_proposer = assign_proposer(validators, 1, epoch_seed)
slot_2_proposer = assign_proposer(validators, 2, epoch_seed)
assert slot_1_proposer.name != slot_2_proposer.name

block_1 = Block(
    1, network.mempool_ids(slot_1_proposer.name), genesis.hash, slot_1_proposer.name
)
proposal_time_views = {
    "NUS-Node": {genesis.hash},
    "SketchyGuy-Node": {genesis.hash, block_1.hash},
    "Emma-Node": {genesis.hash, block_1.hash},
    "Farid-Node": {genesis.hash, block_1.hash},
}
assert block_1.hash not in proposal_time_views[slot_2_proposer.name]
block_2_alt = Block(
    2, network.mempool_ids(slot_2_proposer.name), genesis.hash, slot_2_proposer.name
)

print(f"Slot 1 proposer: {slot_1_proposer.name} creates Block #1.")
print(
    f"Slot 2 proposer: {slot_2_proposer.name} has NOT received Block #1; "
    "it builds on Genesis."
)
print("\nPropagation fork (both arrows leave Genesis):")
print("          Genesis")
print("     +----+----+")
print("     |         |")
print("   Block #1    Block #2-alt")
print(f"   {slot_1_proposer.name:<12} {slot_2_proposer.name}")


## 6. Attestations turn local views into fork-choice weight

After propagation continues, validators attest to the candidate tip they have received. These votes are explicit observations, not random outcomes. Our tiny fork-choice model adds the stake behind each candidate and deliberately rejects ties, because a real protocol needs a specified tie-break rule too.


In [ ]:
def attestation_weight(
    tip_hash: str,
    votes: dict[str, str],
    validators_by_name: dict[str, Validator],
) -> int:
    unknown = set(votes) - set(validators_by_name)
    if unknown:
        raise ValueError(f"Unknown attesters: {sorted(unknown)}")
    return sum(
        validators_by_name[name].stake
        for name, voted_hash in votes.items()
        if voted_hash == tip_hash
    )


def fork_choice_two_tips(
    candidate_tips: list[Block],
    votes: dict[str, str],
    validators_by_name: dict[str, Validator],
) -> tuple[Block, dict[str, int]]:
    candidate_hashes = {block.hash for block in candidate_tips}
    invalid_votes = set(votes.values()) - candidate_hashes
    if invalid_votes:
        raise ValueError("An attestation names an unknown candidate tip.")
    weights = {
        block.hash: attestation_weight(block.hash, votes, validators_by_name)
        for block in candidate_tips
    }
    if len(set(weights.values())) != len(weights):
        raise ValueError("Teaching model requires an explicit tie-break rule.")
    winner_hash = max(weights, key=weights.get)
    return next(block for block in candidate_tips if block.hash == winner_hash), weights


attestation_time_views = {
    "NUS-Node": block_1.hash,
    "SketchyGuy-Node": block_2_alt.hash,
    "Emma-Node": block_1.hash,
    "Farid-Node": block_2_alt.hash,
}
votes = dict(attestation_time_views)

print("\nView-derived attestations:")
for name, voted_hash in votes.items():
    label = "Block #1" if voted_hash == block_1.hash else "Block #2-alt"
    print(f"  {name} received {label} and votes for {label}.")

winner, weights = fork_choice_two_tips(
    [block_1, block_2_alt], votes, validators_by_name
)
print(f"Block #1 stake weight: {weights[block_1.hash]}")
print(f"Block #2-alt stake weight: {weights[block_2_alt.hash]}")
print("Canonical tip: Block #1")


**Read the result:** Block #1 wins because NUS-Node and Emma-Node contribute 170 stake, versus 130 stake for Block #2-alt. Fork choice follows the attestation weight behind each received view, not the block's slot number.


## 7. PoW and PoS: keep three questions separate

| Question | Proof of work | Proof of stake in this model |
| --- | --- | --- |
| Who may propose? | Any miner that wins the hash puzzle race. | One scheduled proposer for the slot. |
| Why can a fork appear? | Competing miners can find blocks near the same time. | A scheduled proposer can still be missing a recently propagated block. |
| How is a tip chosen? | More cumulative proof of work. | More stake-weighted attestations. |

The lesson is not that PoS eliminates networking. It changes the permission to propose and the fork-choice evidence, while ordinary delay can still give honest participants different local views. The next notebook will use this boundary between off-chain facts and on-chain agreement to study oracles.
